
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# 델타 테이블 랩 수집 및 조작

이 Notebook은 Delta Lake가 데이터 레이크하우스에 제공하는 몇 가지 기능에 대한 실습 검토를 제공합니다.

## 환경 준비 (Databricks Free Edition)

- **카탈로그** `db_catalog`, **스키마** `db_schema` 를 사용합니다.
- 스키마 `db_schema` 아래 **`myfiles`** 볼륨을 만들고, **`employees.csv`** 를 업로드합니다.
- (다른 노트북에서 안내한 것과 동일한 데이터 레이아웃을 사용합니다.)

In [0]:
%sql
USE CATALOG db_catalog;
USE SCHEMA db_schema;

## 실습 시작

1. 현재 카탈로그를 **db_catalog**로 설정하고 스키마를 특정 스키마로 설정합니다.

    **힌트**:
    - 카탈로그: `USE CATALOG`
    - 스키마: `db_schema (or you can hardcode your schema name)`

In [0]:
USE CATALOG db_catalog;
USE SCHEMA db_schema;

In [0]:
%skip
USE CATALOG db_catalog;
USE SCHEMA db_schema;

2. 쿼리를 실행하여 현재 카탈로그 및 스키마를 봅니다. 결과에 모듈의 카탈로그(**db_catalog**) 및 특정 스키마가 표시되는지 확인합니다.

In [0]:
SELECT current_catalog(), current_schema()

In [0]:
%skip
SELECT current_catalog(), current_schema()

3. 스키마에서 사용 가능한 볼륨을 보시고 **myfiles** 볼륨이 나열되어 있는지 확인합니다.

In [0]:
SHOW VOLUMES;

In [0]:
%skip
SHOW VOLUMES;

4. 볼륨의 **myfiles** 파일을 나열하시고 **name** 열을 확인하여 볼륨에 저장된 파일 유형을 확인합니다. 밑줄(_)으로 시작하는 추가 파일은 무시합니다.

**힌트**: 다음 경로 형식을 사용하여 볼륨에 액세스합니다: */Volumes/catalog_name/schema_name/volume_name/*.

In [0]:
%python
spark.sql(f"LIST '/Volumes/db_catalog/db_schema/myfiles'").display()

In [0]:
%skip
%python
spark.sql(f"LIST '/Volumes/db_catalog/db_schema/myfiles'").display()

5. 볼륨 경로를 직접 쿼리하고 적절한 파일 형식을 사용하여 파일의 데이터를 미리 봅니다. 볼륨 경로 주위에 백틱을 사용해야 합니다.

**힌트**: SELECT * FROM \<file-format\>. \`\<path-to-volume-myfiles\>\`

In [0]:
%python
spark.sql(f'''SELECT *
FROM csv.`/Volumes/db_catalog/db_schema/myfiles`
LIMIT 10
''').display()

In [0]:
%skip
%python
spark.sql(f'''SELECT *
FROM csv.`/Volumes/db_catalog/db_schema/myfiles`
LIMIT 10
''').display()

6. 스키마에 다음 열이 포함된 **employees_bronze** 테이블을 만듭니다.
| 필드 이름 | 필드 유형 |
| --- | --- |
| ID | INT |
| FirstName | STRING |
| Country | STRING |
| Role | STRING |

**참고:** DROP TABLE 문은 오류를 방지하기 위해 이미 존재하는 경우 테이블을 삭제합니다.

In [0]:
DROP TABLE IF EXISTS employees_bronze;

CREATE TABLE IF NOT EXISTS employees_bronze (
  ID INT,
  FirstName STRING,
  Country STRING,
  Role STRING
);

In [0]:
%skip
DROP TABLE IF EXISTS employees_bronze;

CREATE TABLE IF NOT EXISTS employees_bronze (
  ID INT,
  FirstName STRING,
  Country STRING,
  Role STRING
);

7. [COPY INTO](https://docs.databricks.com/en/sql/language-manual/delta-copy-into.html) 문을 사용하여 **myfiles** 볼륨의 CSV 파일을 **employees_bronze** 테이블에 채웁니다. 다음 옵션을 포함합니다.
    - FROM 경로: `/Volumes/db_catalog/db_schema/myfiles/`
    - FILEFORMAT = CSV
    - FORMAT_OPTIONS: header, inferSchema 등

    행 수가 **employees.csv** 의 레코드 수와 일치하는지 확인합니다.

In [0]:
%python
spark.sql(f'''COPY INTO employees_bronze
  FROM '/Volumes/db_catalog/db_schema/myfiles'
  FILEFORMAT = CSV
  FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
  ''').display()

In [0]:
%skip
%python
spark.sql(f'''COPY INTO employees_bronze
  FROM '/Volumes/db_catalog/db_schema/myfiles'
  FILEFORMAT = CSV
  FORMAT_OPTIONS ('header' = 'true', 'inferSchema' = 'true')
  ''').display()

8. **employees_bronze** 테이블의 행 수를 계산합니다. **employees.csv**( 및 있다면 추가로 적재한 CSV)의 내용과 행 수가 기대와 맞는지 확인합니다.

In [0]:
SELECT count(*) as totalrows
FROM employees_bronze;

In [0]:
%skip
SELECT count(*) as totalrows
FROM employees_bronze;

9. **employees_bronze** 테이블의 기록을 봅니다. 버전 0 및 버전 1을 사용할 수 있는지 확인합니다.

In [0]:
DESCRIBE HISTORY employees_bronze;

In [0]:
%skip
DESCRIBE HISTORY employees_bronze;

10. 다음 스크립트는 조건에 맞는 일부 행을 삭제합니다. (예: 실습에서 추가한 `employees2.csv`의 행만 제거합니다.) 삭제 전·후 행 수를 비교해 보세요.

In [0]:
DELETE FROM employees_bronze
WHERE ID IN (99, 100);

11. **employees_bronze** 테이블의 기록을 봅니다. **operation** 열을 봅니다. *DELETE* 작업이 발생한 버전을 봅니다.

In [0]:
DESCRIBE HISTORY employees_bronze;

In [0]:
%skip
DESCRIBE HISTORY employees_bronze;

12. 쿼리를 실행하여 현재 버전의 테이블에 있는 총 행 수를 계산합니다 **employees_bronze** . 현재 테이블에 16,545 *개의 행이 포함되어* 있는지 확인합니다.

**힌트:** 기본적으로 최신 버전이 사용됩니다.

In [0]:
SELECT count(*) AS totalrows
FROM employees_bronze;

In [0]:
%skip
SELECT count(*) AS totalrows
FROM employees_bronze;

13. 테이블의 이전 버전을 쿼리하여 DELETE 직전 시점의 행 수와 비교합니다. `VERSION AS OF` 구문을 사용합니다.

**힌트:** FROM \<table> VERSION AS OF \<n>

In [0]:
SELECT count(*) AS totalrows
FROM employees_bronze VERSION AS OF 1;

In [0]:
%skip
SELECT count(*) AS totalrows
FROM employees_bronze VERSION AS OF 1;

**도전**


14. 아차! 실수로 이전의 행을 삭제했습니다. [RESTORE](https://docs.databricks.com/en/sql/language-manual/delta-restore.html) 문을 사용하여 Delta 테이블을 *DELETE* 작업 이전의 원래 상태로 복원합니다.

In [0]:
RESTORE TABLE employees_bronze TO VERSION AS OF 1;

In [0]:
%skip
RESTORE TABLE employees_bronze TO VERSION AS OF 1;

15. **employees_bronze** 테이블의 이력을 봅니다. 최신 버전에 *RESTORE* **operation** 포함되어 있는지 확인합니다.

In [0]:
DESCRIBE HISTORY employees_bronze;

In [0]:
%skip
DESCRIBE HISTORY employees_bronze;

16. 현재 **employees_bronze** 테이블의 총 행 수를 계산합니다. RESTORE 이후 행 수가 DELETE 이전 버전과 같아졌는지 확인합니다.

In [0]:
SELECT count(*) as totalrows
FROM employees_bronze;

In [0]:
%skip
SELECT count(*) as totalrows
FROM employees_bronze;

17. **employees_bronze** 테이블을 삭제합니다.

In [0]:
DROP TABLE IF EXISTS employees_bronze;

In [0]:
%skip
DROP TABLE IF EXISTS employees_bronze;

### 요약
이 실습을 완료하면 이제 익숙해질 수 있습니다.
* 표준 Delta Lake 테이블 생성 및 데이터 조작 명령 완료
* 테이블 이력을 포함한 테이블 메타데이터 검토
* 스냅샷 쿼리 및 롤백을 위해 Delta Lake 버전 관리 활용

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>